# Hermite/Mehler derivation check — closed-form order-degradation curve
Verifies the derived closed form
$$F(\rho) = \frac{(1-\rho^2)^2}{(1+\rho^2)(1+2\rho^2)}$$
three ways: (1) semi-analytic machinery must equal it to machine precision for the monomial;
(2) the same machinery must *predict* the tanh_prod curve (validated against the pinned RBF artifact);
(3) an independent Monte Carlo with the explicitly derived optimal additive function.
Outputs to `MyDrive/KDD_Interactions/results/hermite_verification/`.


In [ ]:
# Cell 1 — Mount Drive and set up output folder
from google.colab import drive
drive.mount('/content/drive')
import os
BASE = '/content/drive/MyDrive/KDD_Interactions'
OUT = os.path.join(BASE, 'results', 'hermite_verification')
os.makedirs(OUT, exist_ok=True)
print('output folder:', OUT)


In [ ]:
# Cell 2 — Derivation machinery (normalized probabilists' Hermite, Mehler decoupling)
import numpy as np, json, csv, time, hashlib
from numpy.polynomial.hermite_e import hermegauss

def herm_norm(x, N):
    H = np.zeros((N + 1, len(x)))
    H[0] = 1.0
    if N >= 1: H[1] = x
    for n in range(1, N):
        H[n + 1] = (x * H[n] - np.sqrt(n) * H[n - 1]) / np.sqrt(n + 1)
    return H

NODES, WTS = hermegauss(240)
WTS = WTS / np.sqrt(2 * np.pi)
NMAX = 60
HN = herm_norm(NODES, NMAX)

def semi_analytic_F(f, rho, nmax=NMAX):
    """Irreducible fraction for h = f(x1) f(x2) f(x3), E[f]=0, probe geometry
    (x2 independent; x3 = rho x1 + sqrt(1-rho^2) z). Sector reduction kills S13;
    Mehler decouples the additive normal equations per Hermite degree."""
    t = HN @ (WTS * f(NODES))
    m = (t * rho ** np.arange(nmax + 1)) @ HN          # E[f(x3)|x1] on grid
    c = HN @ (WTS * (f(NODES) * m))                    # coeffs of E[g|x1]
    captured = np.sum(2 * c[1:] ** 2 / (1 + rho ** np.arange(1, nmax + 1)))
    Eg = np.sum(t ** 2 * rho ** np.arange(nmax + 1))
    X1 = NODES[:, None]; Z = NODES[None, :]
    X3 = rho * X1 + np.sqrt(max(0.0, 1 - rho ** 2)) * Z
    W2 = WTS[:, None] * WTS[None, :]
    Eg2 = np.sum(W2 * f(X1) ** 2 * f(X3) ** 2)
    return (Eg2 - Eg ** 2 - captured) / Eg2

closed_form = lambda r: (1 - r**2) ** 2 / ((1 + r**2) * (1 + 2 * r**2))
RHOS = [0.0, 0.3, 0.5, 0.7, 0.9, 0.99, 1.0]
CODE_SHA = hashlib.sha256(b"".join(f.__code__.co_code for f in [herm_norm, semi_analytic_F])).hexdigest()
print("code sha256:", CODE_SHA)


In [ ]:
# Cell 3 — Three-way verification, results to Drive
rows, checks = [], []

# (1) machinery == closed form for the monomial (machine precision)
for r in RHOS:
    sa, cf = semi_analytic_F(lambda x: x, r), closed_form(r)
    rows.append({"experiment": "hermite_verification", "check": "machinery_vs_closed",
                 "rho": r, "value": sa, "reference": cf, "diff": abs(sa - cf)})
checks.append(("machinery == closed form (all rho, <1e-12)",
               all(x["diff"] < 1e-12 for x in rows)))

# (2) prediction of the tanh_prod curve vs pinned RBF artifact (upper-bound direction)
PINNED_TANH = {0.0: 1.0114, 0.3: 0.7827, 0.5: 0.4884, 0.7: 0.2086,
               0.9: 0.0266, 0.99: 0.0004, 1.0: 0.0001}
tanh_ok = True
for r in RHOS:
    pred = semi_analytic_F(np.tanh, r)
    piv = PINNED_TANH[r]
    rows.append({"experiment": "hermite_verification", "check": "tanh_prediction_vs_pinned_rbf",
                 "rho": r, "value": pred, "reference": piv, "diff": piv - pred})
    if not (-0.002 < piv - pred < 0.015):   # RBF should sit slightly ABOVE prediction
        tanh_ok = False
checks.append(("tanh prediction: pinned RBF within (-0.002, +0.015) above prediction", tanh_ok))

# (3) independent Monte Carlo with the derived optimal additive function
rng = np.random.default_rng(7)
mc_ok = True
for r in [0.3, 0.5, 0.9]:
    n = 4_000_000
    x1, x2, z = rng.standard_normal((3, n))
    x3 = r * x1 + np.sqrt(1 - r**2) * z
    h = x1 * x2 * x3
    s = r / (1 + r**2)
    fit = x2 * (r + s * ((x1**2 - 1) + (x3**2 - 1)))
    frac = float(np.mean((h - fit) ** 2) / np.mean(h ** 2))
    rows.append({"experiment": "hermite_verification", "check": "mc_optimal_additive",
                 "rho": r, "value": frac, "reference": closed_form(r), "diff": abs(frac - closed_form(r))})
    if abs(frac - closed_form(r)) > 0.005:
        mc_ok = False
checks.append(("Monte Carlo matches closed form within 0.005", mc_ok))

import csv, json, time, os
with open(os.path.join(OUT, "results.csv"), "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
with open(os.path.join(OUT, "metadata.json"), "w") as f:
    json.dump({"experiment": "hermite_verification", "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
               "code_sha256": CODE_SHA, "numpy": np.__version__,
               "closed_form": "(1-r^2)^2 / ((1+r^2)(1+2r^2))",
               "pinned_tanh_source": "order_probe_v2 results.csv, frac_rbf_ridge"}, f, indent=2)

print(f"{'check':40s}{'rho':>6s}{'value':>12s}{'reference':>12s}{'diff':>12s}")
for x in rows:
    print(f"{x['check']:40s}{x['rho']:6.2f}{x['value']:12.6f}{x['reference']:12.6f}{x['diff']:12.2e}")
print()
story = []
for name, ok in checks:
    line = ("PASS  " if ok else "FAIL  ") + name
    story.append(line); print(line)
with open(os.path.join(OUT, "check.txt"), "w") as f:
    f.write("\n".join(story) + "\n")
print("\nwrote results.csv, metadata.json, check.txt")
